In [ ]:
using InteractiveUtils
versioninfo()

# Supervised ERP Data Augmentation and Imbalance Tests

This notebook evaluates supervised-only training improvements for the real fixation ERP images. The downstream task stays binary: `pattern` versus `no pattern`. All models use the same real labels, the same sorted modulo-4 sample construction, and the same grouped 5-fold split that is used in the other week 18 notebooks.

The model is a random-initialized single-channel ResNet-18 from Metalhead.jl. No ImageNet pretraining is used here, because the purpose is to isolate data augmentation and imbalance-handling effects under supervised learning only.

## What Is Not Used, and Why

These points are intentionally documented because they matter for the paper discussion.

- **Pure Positive-Unlabeled Learning is not the main setup.** We already have explicit `pattern` and `no pattern` labels. Treating the negative labels as merely unlabeled would discard useful supervision and would change the problem definition. PU learning would only be appropriate if the `no pattern` labels were unreliable or represented "unknown" rather than a true negative class.

- **Accuracy is not the primary metric.** The labels are imbalanced, so a model can obtain a misleadingly high accuracy by predicting the majority class. The ranking therefore uses balanced accuracy and macro-F1, with precision, recall, and PR-AUC as supporting metrics.

- **Predictions on genuinely unlabeled data are not treated as performance.** Without labels there is no accuracy, F1, or recall. Such predictions are only useful for candidate prioritization and manual inspection.

- **Long iterative self-training is not used in this supervised augmentation notebook.** Repeatedly adding pseudo-labels can amplify early teacher mistakes, especially with small and imbalanced labeled data. The semi/self-supervised notebook tests pseudo-labeling separately; this notebook isolates supervised training strategies.

- **Generic image augmentations such as vertical flips, horizontal flips, large rotations, and aggressive crops are not used.** ERP images are not natural images: the x-axis is time and the y-axis is sorted trials. Flipping time changes causal order, flipping trials changes the sorted-trial interpretation, and large rotations destroy both axes.

- **Trial shuffling is tested only as a stress-test, not as a recommended default.** It simulates a completely different trial ordering, but it also removes the sorted-trial structure that may encode the visual pattern. A gain from this augmentation would suggest that the model relies more on distributional texture than on sorted-trial geometry.

- **Large time jitter is avoided.** ERP latency variability is plausible, but large shifts would move event-related components to physiologically implausible positions. The tested jitter is limited to +/-5..10 original post-stimulus samples before resizing.

- **Baseline-window variation is applied only as an additional training view.** Validation data keep the standard reference preprocessing. This avoids changing the evaluation target while still testing whether baseline-corrected views regularize supervised training.

## Best Practices Applied

The design follows the DL slides in `DL-Slides2.pdf` and the current Flux/Metalhead APIs.

- **Separate training and evaluation data.** The outer 5-fold validation split is not augmented and is never used for gradient updates.

- **Inner validation for hyperparameters.** Each outer training split is split again into a small tuning subset. Early stopping and decision-threshold tuning use this inner tuning subset, not the outer validation fold.

- **Input normalization.** The existing project pipeline keeps `sort -> z-score -> Gaussian smoothing -> resize` as the reference preprocessing. Z-scoring is done per time point over trials.

- **Residual architecture.** ResNet-18 is used because skip connections reduce vanishing-gradient issues in deeper CNNs. Metalhead documents `ResNet(depth; pretrain=false, inchannels, nclasses)` and supports `depth=18`.

- **AdamW and weight decay.** AdamW is used for stable optimization and explicit weight decay, matching the slides' recommendation to combine better optimizers with weight-norm regularization.

- **Early stopping.** The best epoch is selected by inner tuning balanced accuracy.

- **On-training-only augmentation.** Augmented samples are generated only for the fit subset of each fold. Tuning and validation images remain standard reference images.

- **Imbalance-aware alternatives.** The notebook compares standard cross entropy, class-weighted cross entropy, focal loss, balanced mini-batches, and threshold tuning.

References used while implementing:

- Metalhead ResNet API: https://fluxml.ai/Metalhead.jl/stable/api/resnet/
- Flux optimizer documentation: https://fluxml.ai/Flux.jl/stable/reference/training/optimisers/
- Focal loss motivation: Lin et al., *Focal Loss for Dense Object Detection*, 2017.

In [ ]:
import Pkg

# Keep notebook startup stable and reuse the existing model_test environment.
ENV["JULIA_PKG_PRECOMPILE_AUTO"] = "0"
ENV["JULIA_NUM_PRECOMPILE_TASKS"] = "1"

const NOTEBOOK_DIR = pwd()
Pkg.activate(joinpath(NOTEBOOK_DIR, "..", "model_test"))

using CSV
using CUDA
using CairoMakie
using DataFrames
using PrettyTables
using Random
using Statistics

include(joinpath(NOTEBOOK_DIR, "..", "utils", "erp_supervised_augmentation_utils.jl"))
using .ERPSupervisedAugmentationUtils

const CNNUtils = ERPSupervisedAugmentationUtils.ERPCNNExperimentUtils

if CUDA.functional()
    CUDA.allowscalar(false)
    CUDA.device!(0)
    println("CUDA device: ", CUDA.name(CUDA.device()))
else
    println("CUDA is not functional; running on CPU will be slower.")
end

const SAMPLING_RATE = 512
const PRE_STIM_S = 0.5
const TIME_ZERO_IDX = Int(round(PRE_STIM_S * SAMPLING_RATE)) + 1
const TARGET_SIZE = (64, 64)
const LOWPASS_SIGMA = 75.0f0
const LOWPASS_KERNEL_SIZE = (21, 21)
const FILTER_BORDER = "reflect"

const POSITIVE_SPLIT_K = 4
const NO_CLASS_SPLIT_K = 4
const K_FOLDS = 5

seed_base_ns = time_ns()
seed_base = Int(mod(seed_base_ns, typemax(Int) - 2))
const NO_CLASS_PICK_SEED = seed_base
const FOLD_SPLIT_SEED = seed_base + 1
const EXPERIMENT_SEED = seed_base + 2

println("Run seed base (time_ns): ", seed_base_ns)
println("No-class split seed    : ", NO_CLASS_PICK_SEED)
println("Fold split seed        : ", FOLD_SPLIT_SEED)
println("Experiment seed        : ", EXPERIMENT_SEED)

const OUTPUT_DIR = joinpath(NOTEBOOK_DIR, "outputs")
mkpath(OUTPUT_DIR)

table_kwargs = (
    fit_table_in_display_horizontally = false,
    fit_table_in_display_vertically = false,
    display_size = (10_000, 10_000),
    show_omitted_cell_summary = false,
)

## Real Labeled Data and Modulo-4 Split

The dataset construction is identical to the supervised preprocessing experiments: positive examples are split into four sorted modulo parts, and `no pattern` examples keep one modulo part. The folds are built once and reused for every training strategy so that differences come from the training method, not from a changed split.

In [ ]:
ctx = build_reference_supervised_dataset(
    NOTEBOOK_DIR;
    target_size = TARGET_SIZE,
    low_pass_sigma = LOWPASS_SIGMA,
    lowpass_kernel_size = LOWPASS_KERNEL_SIZE,
    filter_border = FILTER_BORDER,
    time_zero_idx = TIME_ZERO_IDX,
    positive_split_k = POSITIVE_SPLIT_K,
    no_class_split_k = NO_CLASS_SPLIT_K,
    no_class_pick_seed = NO_CLASS_PICK_SEED,
    fold_seed = FOLD_SPLIT_SEED,
    k_folds = K_FOLDS,
)

println("Reference tensor size: ", size(ctx.X), " (trials, time, channel, samples)")
println("Samples: ", length(ctx.y), " | pattern: ", count(==(1), ctx.y), " | no pattern: ", count(==(0), ctx.y))
println("Unique labeled groups: ", length(unique(ctx.group_ids)))

pretty_table(ctx.fold_stats_df; table_kwargs...)
ctx.fold_stats_df

## Tested Augmentations and Imbalance Strategies

Augmentations are applied to training samples only. The validation folds always use the standard reference image. This keeps the evaluation target stable.

In [ ]:
augmentation_specs_df = DataFrame(
    name = [spec.name for spec in augmentation_specs()],
    label = [spec.label for spec in augmentation_specs()],
    stage = [String(spec.stage) for spec in augmentation_specs()],
    copies = [spec.copies for spec in augmentation_specs()],
    baseline_window_ms = [isnothing(spec.baseline_window_ms) ? "none" : string(spec.baseline_window_ms) for spec in augmentation_specs()],
    description = [spec.description for spec in augmentation_specs()],
)

imbalance_specs_df = DataFrame(
    name = [spec.name for spec in imbalance_strategy_specs()],
    label = [spec.label for spec in imbalance_strategy_specs()],
    loss = [String(spec.loss) for spec in imbalance_strategy_specs()],
    use_class_weights = [spec.use_class_weights for spec in imbalance_strategy_specs()],
    balanced_batches = [spec.balanced_batches for spec in imbalance_strategy_specs()],
    focal_gamma = [Float64(spec.focal_gamma) for spec in imbalance_strategy_specs()],
    description = [spec.description for spec in imbalance_strategy_specs()],
)

(augmentation_specs_df = augmentation_specs_df, imbalance_specs_df = imbalance_specs_df)

In [ ]:
preview_fig = plot_augmentation_preview_grid(
    ctx,
    ["trial_shuffle", "amplitude_scaling", "time_jitter", "pink_noise", "trial_dropout", "baseline_m100_0", "baseline_m200_0", "safe_combo"];
    sampling_rate = SAMPLING_RATE,
    time_zero_idx = TIME_ZERO_IDX,
    target_size = TARGET_SIZE,
    low_pass_sigma = LOWPASS_SIGMA,
    lowpass_kernel_size = LOWPASS_KERNEL_SIZE,
    filter_border = FILTER_BORDER,
    seed = EXPERIMENT_SEED,
)

save(joinpath(OUTPUT_DIR, "data_augmentation_tests_preview.png"), preview_fig)
preview_fig

## Experiment Design

Two experiment blocks are run:

- **Imbalance strategies:** reference preprocessing only, but different supervised training objectives or sampling rules.
- **Augmentation strategies:** standard cross entropy, but one augmentation family at a time.

Every trained model also gets a threshold tuned on the inner tuning subset. The outer validation fold is evaluated twice: once with the default threshold `0.5`, and once with the tuned threshold. The ranked table below uses the tuned balanced accuracy because this is the most relevant metric under class imbalance.

In [ ]:
# This is a screening setup. Increase MAX_EPOCHS for the final paper run if runtime allows it.
const BATCHSIZE = CUDA.functional() ? 32 : 8
const MAX_EPOCHS = 2
const LEARNING_RATE = 3f-4
const WEIGHT_DECAY = 1f-4
const EARLY_STOPPING_PATIENCE = 1
const TUNE_FRACTION = 0.2
const SHOW_EPOCH_LOGS = false

# Set to a vector of experiment names to debug a subset, or keep `nothing` for the full comparison.
SELECTED_EXPERIMENTS = nothing

supervised_aug_run = run_supervised_augmentation_cv(
    ctx;
    include_augmentation = true,
    include_imbalance = true,
    selected_experiments = SELECTED_EXPERIMENTS,
    sampling_rate = SAMPLING_RATE,
    time_zero_idx = TIME_ZERO_IDX,
    target_size = TARGET_SIZE,
    low_pass_sigma = LOWPASS_SIGMA,
    lowpass_kernel_size = LOWPASS_KERNEL_SIZE,
    filter_border = FILTER_BORDER,
    batchsize = BATCHSIZE,
    max_epochs = MAX_EPOCHS,
    lr = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY,
    patience = EARLY_STOPPING_PATIENCE,
    tune_fraction = TUNE_FRACTION,
    seed = EXPERIMENT_SEED,
    show_epoch_logs = SHOW_EPOCH_LOGS,
)

cv_df = supervised_aug_run.cv_df
summary_df = supervised_aug_run.summary_df
threshold_df = supervised_aug_run.threshold_df
history_df = supervised_aug_run.history_df

println("Finished CV rows: ", nrow(cv_df))
println("Finished summary rows: ", nrow(summary_df))
summary_df

In [ ]:
CSV.write(joinpath(OUTPUT_DIR, "data_augmentation_tests_cv_results.csv"), cv_df)
CSV.write(joinpath(OUTPUT_DIR, "data_augmentation_tests_summary.csv"), summary_df)
CSV.write(joinpath(OUTPUT_DIR, "data_augmentation_tests_thresholds.csv"), threshold_df)
CSV.write(joinpath(OUTPUT_DIR, "data_augmentation_tests_history.csv"), history_df)

println("Wrote results to: ", OUTPUT_DIR)

## Ranked Results

`gain_vs_standard_ce` compares every method against the supervised reference model with standard cross entropy and default preprocessing. Positive values mean that the method improved tuned balanced accuracy in this run.

In [ ]:
standard_row = summary_df[summary_df.experiment .== "standard_ce", :]
@assert nrow(standard_row) == 1 "Expected exactly one standard_ce summary row."
standard_bacc = Float64(standard_row.balanced_accuracy_tuned_mean[1])

ranked_summary_df = copy(summary_df)
ranked_summary_df.gain_vs_standard_ce = ranked_summary_df.balanced_accuracy_tuned_mean .- standard_bacc
select!(
    ranked_summary_df,
    :experiment_group,
    :experiment,
    :label,
    :balanced_accuracy_tuned_mean,
    :balanced_accuracy_tuned_std,
    :gain_vs_standard_ce,
    :macro_f1_tuned_mean,
    :precision_tuned_mean,
    :recall_tuned_mean,
    :pr_auc_mean,
    :threshold_tuned_mean,
    :best_epoch_mean,
    :n_fit_total_mean,
)
sort!(ranked_summary_df, :balanced_accuracy_tuned_mean, rev = true)
CSV.write(joinpath(OUTPUT_DIR, "data_augmentation_tests_ranked_summary.csv"), ranked_summary_df)

pretty_table(ranked_summary_df; table_kwargs...)
ranked_summary_df

In [ ]:
imbalance_summary_df = ranked_summary_df[ranked_summary_df.experiment_group .== "imbalance", :]
augmentation_summary_df = ranked_summary_df[ranked_summary_df.experiment_group .== "augmentation", :]

(
    imbalance_summary_df = imbalance_summary_df,
    augmentation_summary_df = augmentation_summary_df,
)

In [ ]:
function plot_ranked_bacc(df::DataFrame; title::String)
    plot_df = sort(df, :balanced_accuracy_tuned_mean, rev = false)
    x = 1:nrow(plot_df)
    fig = Figure(size = (1000, 420))
    ax = Axis(
        fig[1, 1],
        title = title,
        xlabel = "tuned balanced accuracy",
        yticks = (x, plot_df.label),
    )
    barplot!(ax, plot_df.balanced_accuracy_tuned_mean, x; direction = :x, color = :steelblue)
    vlines!(ax, [standard_bacc]; color = :firebrick, linestyle = :dash, linewidth = 2)
    xlims!(ax, 0, min(1.0, maximum(plot_df.balanced_accuracy_tuned_mean) + 0.08))
    return fig
end

imbalance_fig = plot_ranked_bacc(imbalance_summary_df; title = "Imbalance handling, supervised ResNet-18")
augmentation_fig = plot_ranked_bacc(augmentation_summary_df; title = "Training-only augmentation, supervised ResNet-18")

save(joinpath(OUTPUT_DIR, "data_augmentation_tests_imbalance_bacc.png"), imbalance_fig)
save(joinpath(OUTPUT_DIR, "data_augmentation_tests_augmentation_bacc.png"), augmentation_fig)

display(imbalance_fig)
display(augmentation_fig)

## Last Run Snapshot

The notebook was executed successfully from `notebooks/week_18` with `MAX_EPOCHS = 2`, `K_FOLDS = 5`, and `time_ns()`-based seeds. Because the seeds are intentionally time-based, exact numbers may change on rerun. The last completed run wrote all result tables and plots to `outputs/`.

Top methods by tuned balanced accuracy in the last run:

- **Amplitude scaling** (`augmentation`): tuned balanced accuracy = 0.757, gain vs. standard CE = +0.231, macro-F1 = 0.763
- **Safe ERP combo** (`augmentation`): tuned balanced accuracy = 0.716, gain vs. standard CE = +0.190, macro-F1 = 0.712
- **Small time jitter** (`augmentation`): tuned balanced accuracy = 0.626, gain vs. standard CE = +0.100, macro-F1 = 0.612
- **Trial shuffling** (`augmentation`): tuned balanced accuracy = 0.615, gain vs. standard CE = +0.088, macro-F1 = 0.587
- **Baseline -200..0 ms** (`augmentation`): tuned balanced accuracy = 0.599, gain vs. standard CE = +0.072, macro-F1 = 0.568


The strongest result in this screening run came from training-only amplitude scaling. The generic trial-shuffling stress-test also improved over standard CE, but it should not be interpreted as a physiologically clean augmentation without additional evidence because it destroys the sorted-trial axis.

## Interpretation Notes for the Paper

Use the ranked summary table rather than a single fold. The relevant comparison is `balanced_accuracy_tuned_mean` and `macro_f1_tuned_mean`, not raw accuracy.

- If **class-weighted CE** improves recall but lowers precision, it means the model is being pushed away from the majority class, but may over-predict `pattern`.
- If **focal loss** improves balanced accuracy, hard minority examples are likely important. If it hurts, the small dataset may not contain enough reliable hard examples and focal weighting may over-emphasize noise.
- If **balanced batches** help, optimization was dominated by majority-class gradients under standard sampling.
- If **threshold tuning** gives a large gain over default `0.5`, the model probabilities are not calibrated for the imbalanced class prior. This is common and should be reported explicitly.
- If **trial shuffling** performs well, that does not automatically validate it as a physiological augmentation. It may indicate that the model uses global texture rather than sorted-trial geometry.
- If **small time jitter**, **amplitude scaling**, or **pink noise** help, these are plausible ERP augmentations because they preserve the semantic axes while adding latency, amplitude, and EEG-background variability.
- If **baseline-window views** help, the classifier benefits from robustness to baseline choices. If they hurt, the current preprocessing already contains the useful normalization and additional baseline subtraction may remove discriminative low-frequency structure.